# 2. Fine-tuning Gemma-3-1B-IT with LoRA

SFT on teacher-generated expressions using Unsloth + TRL.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --upgrade -q
!pip install trl transformers datasets accelerate bitsandbytes peft -q

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

print(f"torch: {torch.__version__}")
print(f"CUDA:  {torch.cuda.is_available()}")
print(f"GPU:   {torch.cuda.get_device_name(0)}")

## Load and prepare dataset

In [ ]:
df = pd.read_csv("qwen3_full_results.csv", on_bad_lines="skip")
print(f"Loaded {len(df)} rows")
print(df.head())


def create_training_example(row):
    nums_str = str(row["nums"]).strip("[]").split()
    nums_formatted = ", ".join(nums_str)
    return (
        f"<start_of_turn>user\n"
        f"Given the target number {row['target']} and the list of numbers [{nums_formatted}], "
        f"find a mathematical expression using basic operations (+, -, *, /) that equals the target. "
        f"You can use each number exactly once.\n"
        f"Provide your answer as a mathematical expression.<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{row['expression']}<end_of_turn>"
    )


df["text"] = df.apply(create_training_example, axis=1)
dataset = Dataset.from_pandas(df[["text"]])
print(f"Dataset ready: {len(dataset)} examples")
print(dataset[0]["text"])

## Load Gemma-3-1B-IT with 4-bit quantization

In [ ]:
MAX_SEQ_LENGTH = 2048
LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-3-1b-it",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    gpu_memory_utilization=0.7,
)
print("Base model loaded")

## Apply LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

## Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=TrainingArguments(
        output_dir="./gemma_math_lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Done. Runtime: {stats.metrics['train_runtime']:.0f}s")

## Save adapter

In [ ]:
model.save_pretrained("gemma_math_lora_final")
tokenizer.save_pretrained("gemma_math_lora_final")
print("Saved to ./gemma_math_lora_final")

## Quick sanity check

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = (
    "<start_of_turn>user\n"
    "Given the target number 14 and the list of numbers [3, 13, 6, 8], "
    "find a mathematical expression using basic operations (+, -, *, /) that equals the target. "
    "You can use each number exactly once.\n"
    "Provide your answer as a mathematical expression.<end_of_turn>\n"
    "<start_of_turn>model\n"
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response.split("<start_of_turn>model")[-1].strip())